In [0]:
%run "/Workspace/Users/dungdq.b22kh019@stu.ptit.edu.vn/brigde_and_gateway"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Tổng số cặp similarity: 3225710


Threshold (97th percentile): 0.7350
Số cạnh sau lọc: 98142


Graph: 3254 nodes, 98142 edges


Số communities phát hiện được: 93


Modularity score: 0.6332



Top 10 communities lớn nhất:
community_id
0     621
9     371
28    354
20    279
31    211
46    188
14    183
21    165
7     146
10    118
Name: community_size, dtype: int64


In [0]:
import networkx as nx
import pandas as pd
import numpy as np
from collections import Counter, defaultdict
from itertools import combinations

Đã lưu community results


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Đã khôi phục thành công biến partition!
Số lượng subreddit trong partition: 3254


Tính Gateway scores:
Số communities >= 3 nodes: 51

Gateway tính xong: 45s
Tổng gateway candidates: 3170



Tính Bridge scores
Tổng bridge candidates: 644

Top 10 bridges:
               subreddit  source_community  target_community  bridge_score
603               Barber                75                 9      0.095458
626        2islamist4you                32                 0      0.057774
586      AEWFightForever                47                28      0.043786
604            BlackHair                75                 9      0.033692
605             360Waves                75                 9      0.033534
487           Aliexpress                19                 0      0.033469
486  BehindTheClosetDoor                19                 0      0.033417
588                  ANW                47                 0      0.031051
420        CatholicMemes                15                 0      0.029913
421       AdviceAtheists                15                 0      0.016568


\Phân tích overlap bridge và gateway
Top 100 bridges: 99
Top gateways: 153
Overlap: 4 nodes (4.0%)
Số lượng 4.0% bridges cũng là gateways


Đã lưu kết quả gateway_bridge


In [0]:
G_highway = nx.Graph()
for _, row in edges_pd.iterrows():
    sim = float(row["Similarity_Score"])
    # Inverse weight: khoảng cách = 1 - similarity
    G_highway.add_edge(
        row["Subreddit_A"],
        row["Subreddit_B"],
        weight=1.0 - sim,          # dùng cho shortest path
        similarity=sim              # giữ lại để tham khảo
    )

print(f"Highway graph: {G_highway.number_of_nodes()} nodes, {G_highway.number_of_edges()} edges")

Highway graph: 3254 nodes, 98142 edges


In [0]:
community_representatives = {}
for comm_id, comm_nodes in large_communities.items():
    subgraph = G_highway.subgraph(comm_nodes)
    if subgraph.number_of_nodes() == 0:
        continue
    # Node có degree cao nhất = most connected node
    rep = max(subgraph.nodes(), key=lambda n: G_highway.degree(n))
    community_representatives[comm_id] = rep

print(f"Số community representatives: {len(community_representatives)}")

Số community representatives: 51


In [0]:
print("Tính shortest paths giữa các community pairs")
start_time = datetime.now()

all_shortest_paths = []
comm_list = list(community_representatives.items())
total_pairs = len(comm_list) * (len(comm_list) - 1) // 2

print(f"Tổng số cặp cần tính: {total_pairs}")

for i, (comm_a, rep_a) in enumerate(comm_list):
    for j, (comm_b, rep_b) in enumerate(comm_list):
        if j <= i:
            continue
        if rep_a not in G_highway or rep_b not in G_highway:
            continue
            
        try:
            # Tìm shortest path (dùng Dijkstra với inverse weight)
            path = nx.shortest_path(G_highway, rep_a, rep_b, weight='weight')
            all_shortest_paths.append({
                "community_a": comm_a,
                "community_b": comm_b,
                "path": path,
                "path_length": len(path)
            })
        except nx.NetworkXNoPath:
            pass  

print(f"Tổng paths tìm được: {len(all_shortest_paths)}")
print(f"Thời gian: {(datetime.now()-start_time).seconds}s")

# Thống kê độ dài path
path_lengths = [p["path_length"] for p in all_shortest_paths]
avg_path_len = np.mean(path_lengths)
print(f"Average shortest path length: {avg_path_len:.2f}")
print(f"Distribution: {Counter(path_lengths)}")

Tính shortest paths giữa các community pairs
Tổng số cặp cần tính: 1275
Tổng paths tìm được: 253
Thời gian: 2s
Average shortest path length: 5.24
Distribution: Counter({4: 67, 5: 58, 6: 43, 7: 26, 3: 20, 8: 18, 2: 10, 9: 7, 10: 4})


In [0]:
def extract_subpaths(path, length):
    """Lấy tất cả sub-path có độ dài N từ một path."""
    return [tuple(path[i:i+length]) for i in range(len(path) - length + 1)]

print("\nTrích xuất highways")

highway_results = []

for highway_length in [2, 3, 4, 5]:
    # Đếm tần suất mỗi sub-path
    subpath_counter = Counter()
    total_paths_of_min_length = 0
    
    for path_info in all_shortest_paths:
        path = path_info["path"]
        if len(path) >= highway_length:
            total_paths_of_min_length += 1
            subpaths = extract_subpaths(path, highway_length)
            subpath_counter.update(subpaths)
    
    if total_paths_of_min_length == 0:
        continue
    
    print(f"\n[Length {highway_length}] Total paths: {total_paths_of_min_length}")
    print(f"Unique sub-paths: {len(subpath_counter)}")
    
    top_highways = subpath_counter.most_common(20)
    
    for rank, (subpath, count) in enumerate(top_highways):
        pct = count / total_paths_of_min_length * 100
        
        # Lấy community ID của mỗi node trong highway
        node_communities = [partition.get(node, -1) for node in subpath]
        unique_communities = len(set(node_communities))
        
        highway_results.append({
            "rank": rank + 1,
            "highway_length": highway_length,
            "highway_nodes": " → ".join(subpath),
            "highway_tuple": str(subpath),
            "occurrence_count": count,
            "pct_of_paths": round(pct, 2),
            "pct_of_paths_min_length": round(total_paths_of_min_length / len(all_shortest_paths) * 100, 2),
            "unique_communities_spanned": unique_communities
        })
    
    print(f"Top 5 highways of length {highway_length}:")
    for rank, (subpath, count) in enumerate(top_highways[:5]):
        pct = count / total_paths_of_min_length * 100
        print(f"  {rank+1}. {' → '.join(subpath)} | {count}x ({pct:.1f}%)")


Trích xuất highways

[Length 2] Total paths: 253
Unique sub-paths: 437
Top 5 highways of length 2:
  1. ConservativeKiwi → CapeIndependence | 22x (8.7%)
  2. 2islamist4you → Afghan | 21x (8.3%)
  3. Barber → BeardAdvice | 20x (7.9%)
  4. 90DayFiance → Celebrity_Fantasies2 | 18x (7.1%)
  5. BlackHair → Barber | 17x (6.7%)

[Length 3] Total paths: 243
Unique sub-paths: 510
Top 5 highways of length 3:
  1. BlackHair → Barber → BeardAdvice | 17x (7.0%)
  2. Athleta_gap → BlackestFridayDeals → BestThings | 16x (6.6%)
  3. Aliexpress → Buyee → AmazonSeller | 15x (6.2%)
  4. AimeLeonDore → Athleta_gap → BlackestFridayDeals | 14x (5.8%)
  5. AusMemes → ConservativeKiwi → CapeIndependence | 14x (5.8%)

[Length 4] Total paths: 223
Unique sub-paths: 449
Top 5 highways of length 4:
  1. AimeLeonDore → Athleta_gap → BlackestFridayDeals → BestThings | 14x (6.3%)
  2. AmazonWFShoppers → Aliexpress → Buyee → AmazonSeller | 12x (5.4%)
  3. ButchSelfies → BlackHair → Barber → BeardAdvice | 12x (5.4%)
 

In [0]:
df_highways = pd.DataFrame(highway_results)
print(f"\nTổng số highway entries: {len(df_highways)}")


Tổng số highway entries: 80


In [0]:
print("Top highways:")
top_cross_community = df_highways[df_highways["highway_length"] == 3].nlargest(
    5, "unique_communities_spanned"
)[["highway_nodes", "occurrence_count", "pct_of_paths", "unique_communities_spanned"]]
print(top_cross_community.to_string())

df_highways_spark = spark.createDataFrame(df_highways)
df_highways_spark.coalesce(1).write.mode("overwrite").option("header", True).csv(
    f"wasbs://{container}@{storage_account}.blob.core.windows.net/highway_results"
)
print("\nĐã lưu highway results")

Top highways:
                                       highway_nodes  occurrence_count  pct_of_paths  unique_communities_spanned
20                  BlackHair → Barber → BeardAdvice                17          7.00                           2
21    Athleta_gap → BlackestFridayDeals → BestThings                16          6.58                           2
22                 Aliexpress → Buyee → AmazonSeller                15          6.17                           2
23  AimeLeonDore → Athleta_gap → BlackestFridayDeals                14          5.76                           2
24    AusMemes → ConservativeKiwi → CapeIndependence                14          5.76                           2

Đã lưu highway results
